In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

In [1]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
instance = service.active_account()["instance"]
backend_name = service.least_busy(operational=True, min_num_qubits=16).name

> **Note:** Функції Qiskit є експериментальною можливістю, доступною лише користувачам IBM Quantum&reg; Premium Plan, Flex Plan та On-Prem (через IBM Quantum Platform API) Plan. Вони перебувають у статусі попереднього випуску і можуть змінюватися.

## Огляд
У квантовій хімії задача електронної структури полягає у знаходженні розв'язків електронного рівняння Шрьодінґера — квантових хвильових функцій, що описують поведінку електронів системи. Ці хвильові функції є векторами комплексних амплітуд, де кожна амплітуда відповідає внеску можливої конфігурації електронів.

Основний стан — це хвильова функція системи з найнижчою енергією, яка має особливе значення при вивченні молекулярних систем. Найточніший підхід до обчислення основного стану враховує всі можливі конфігурації електронів, але він стає неможливим для більших систем, оскільки кількість конфігурацій зростає експоненційно зі збільшенням розміру системи.

Ітераційний варіаційний квантовий власний солвер з передачею (HI-VQE) — це інноваційний гібридний квантово-класичний метод для точного оцінювання основного стану молекулярних систем. Він інтегрує квантове апаратне забезпечення з класичними обчисленнями, використовуючи квантові процесори для ефективного дослідження кандидатних конфігурацій електронів та обчислення відповідної хвильової функції на класичних комп'ютерах. Генеруючи компактні, але хімічно точні хвильові функції, HI-VQE сприяє дослідженням і відкриттям у квантовій хімії та матеріалознавстві.

![Зображення, що показує огляд алгоритму HI-VQE від Qunova](../docs/images/guides/qunova-chemistry/overview.svg)

HI-VQE зменшує обчислювальну складність задачі електронної структури, ефективно оцінюючи основний стан з високою точністю. Він зосереджується на ретельно відібраній підмножині найбільш релевантних конфігурацій електронів, оптимізуючи як точність, так і ефективність.

Поєднуючи переваги класичних і квантових комп'ютерів, HI-VQE ітераційно вдосконалює й покращує поточну оцінку хвильової функції. Його унікальні техніки побудови підпростору допомагають зробити вибір конфігурацій ефективнішим, надаючи користувачам більший обчислювальний контроль та покращену точність у квантово-хімічних симуляціях.

Якщо хочеш детальніше дізнатися про алгоритм, можеш [прочитати відповідну дослідницьку статтю](https://arxiv.org/abs/2503.06292).
## Опис
Кількість конфігурацій електронів для молекулярної системи зростає експоненційно зі збільшенням розміру системи. Однак для певних електронних станів, таких як основний стан, зазвичай лише невелика частка конфігурацій суттєво впливає на енергію стану. Методи вибраної конфігураційної взаємодії (SCI) використовують цю розрідженість для зменшення обчислювальних витрат, ідентифікуючи та зосереджуючись на найбільш релевантних конфігураціях. Ця підмножина конфігурацій називається підпростором.

HI-VQE використовує притаманну квантовим комп'ютерам ефективність при поданні молекулярних систем для допомоги в пошуку підпростору. Він інтегрує класичні та квантові підпрограми для розв'язання задачі електронної структури з високою точністю. На відміну від існуючих квантових SCI-методів, HI-VQE поєднує варіаційне навчання, ітераційну побудову підпростору та попереднє діагоналізаційне відсіювання конфігурацій для підвищення ефективності шляхом зменшення кількості квантових вимірювань, ітерацій і витрат на класичну діагоналізацію. Тому HI-VQE можна застосовувати до більших молекулярних систем, які вимагають більше кубітів, і зменшує вартість розв'язання задачі заданого розміру до однакового ступеня точності.

![Зображення з детальним описом кожного кроку в алгоритмі HI-VQE від Qunova.](../docs/images/guides/qunova-chemistry/description.avif)

Для обчислення основного стану системи HI-VQE спочатку використовує класичний хімічний пакет PySCF для генерації молекулярного представлення з наданих користувачем вхідних даних, таких як молекулярна геометрія та інша молекулярна інформація. Потім він входить у гібридний квантово-класичний цикл оптимізації, ітераційно вдосконалюючи підпростір для оптимального представлення основного стану, мінімізуючи кількість включених конфігурацій. Цикл продовжується до виконання критеріїв збіжності, таких як розмір підпростору або стабільність енергії, після чого виводяться обчислена хвильова функція основного стану та енергія. Ці результати можна використовувати для побудови точних поверхонь потенційної енергії та проведення подальшого аналізу системи.

Цикл оптимізації зосереджується на налаштуванні параметрів квантової схеми для генерації якісного підпростору. HI-VQE пропонує три варіанти квантових схем: [`excitation_preserving`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.excitation_preserving), [efficient_su2](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2) та [LUCJ](https://qiskit-community.github.io/ffsim/explanations/lucj.html). Оптимізація ініціалізується поблизу референсного стану Гартрі–Фока завдяки його загальній придатності. Потім схема виконується на квантовому пристрої, і конфігурації вибираються з отриманого квантового стану перед поверненням у вигляді бінарних рядків. Через шуми квантового пристрою деякі відібрані конфігурації можуть бути фізично недійсними, не зберігаючи кількість електронів або спін. HI-VQE вирішує це за допомогою процесу відновлення конфігурацій з пакету [qiskit-addon-sqd](/guides/qiskit-addons-sqd#sample-based-quantum-diagonalization-sqd-overview), щоб користувачі могли або виправити недійсні конфігурації, або відкинути їх.

Дійсні конфігурації потім проходять необов'язковий крок відсіювання для видалення тих, що, за прогнозами, дають мінімальний внесок. Це зменшує розмірність підпростору, тим самим знижуючи вартість кроку діагоналізації. Якщо відсіювання увімкнено, то будується попередній підпросторовий гамільтоніан із дійсних конфігурацій і виконується діагоналізація з дуже м'якими критеріями завершення. Хоча точність отриманих амплітуд для кожної конфігурації є низькою, це ефективно для прогнозування, які конфігурації слід виключити з підпростору на даній ітерації, і обчислюється швидко.

Вибрані конфігурації додаються до підпростору, і гамільтоніан системи проєктується у цей підпростір. Підпростір оновлюється ітераційно, зберігаючи найбільш релевантні конфігурації між ітераціями. Цей підхід відрізняється від альтернативних методів тим, що квантова схема не потребує апроксимації повного основного стану на кожному кроці.

Далі підпросторовий гамільтоніан класично діагоналізується для отримання найменшого власного значення та відповідного власного вектора, що представляють апроксимацію основного стану та його енергії. З покращенням якості підпростору через ітерації обчислений основний стан краще апроксимує істинний основний стан. На цьому етапі може виконуватися додатковий крок відсіювання для видалення з підпростору конфігурацій, що не мають суттєвого внеску в обчислений основний стан. Цей крок забезпечує максимальну компактність підпростору, що переноситься до наступної ітерації. Це оцінюється на основі амплітуд, що повертаються діагоналізацією, оскільки вони представляють важливість внеску кожної конфігурації в обчислений основний стан.

Перевірка збіжності потім визначає, чи подальше навчання покращить результати. Якщо так, виконується необов'язковий крок класичного розширення, параметри квантової схеми оновлюються для подальшої мінімізації обчисленої енергії, і процес повторюється. Крок класичного розширення генерує додаткові конфігурації для підпростору, доповнюючи конфігурації, відібрані з квантового пристрою. Спочатку він визначає конфігурацію з найбільшою амплітудою в результатах діагоналізації, а потім генерує нові конфігурації з одиничними та подвійними збудженнями від визначеної конфігурації. Потрібна кількість цих конфігурацій потім додається до підпростору.

Як тільки визначається, що ітерації збіглись, HI-VQE повертає обчислений основний стан (у формі станів у підпросторі та їхніх амплітуд у хвильовій функції основного стану), його енергію та міру дисперсії енергії, що дає уявлення про те, чи є обчислений стан власним станом гамільтоніана системи.

Користувачі можуть вибирати квантову схему та кількість знімків для кожної квантової схеми, а також контролювати розмір підпростору або вмикати класичну генерацію додаткових конфігурацій на допомогу конфігураціям, згенерованим квантовим способом. Таким чином користувачі можуть налаштовувати поведінку HI-VQE відповідно до своїх цільових застосувань.
## Початок роботи
Спочатку [запроси доступ до функції](https://forms.office.com/r/zN3hvMTqJ1).
Потім автентифікуйся за допомогою свого [API-ключа IBM Quantum&reg;](http://quantum.cloud.ibm.com/) і, якщо ти вже [зберіг свій обліковий запис](/guides/functions#install-qiskit-functions-catalog-client) у локальному середовищі, вибери функцію Qiskit таким чином:

In [ ]:
import reprlib
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("qunova/hivqe-chemistry")

## Вхідні дані
Дивись наступну таблицю з усіма вхідними параметрами, які приймає функція. Наступні розділи [Параметри молекули](#molecule-options) та [Параметри HI-VQE](#hi-vqe-options) містять детальнішу інформацію про ці аргументи.
| Назва                  | Тип                                                            | Опис                                                                                                                                                                                                                                                                                                        | Обов'язковий | За замовчуванням                                                         | Приклад                                                                                   |
|------------------------|----------------------------------------------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|----------|--------------------------------------------------------------------------|-------------------------------------------------------------------------------------------|
| geometry               | Union[List[List[Union[str, Tuple[float, float, float]]]], str] | Може бути рядком або структурованими списками, що містять пари атом–координати. Якщо задано як рядок, це має бути геометрія молекули у форматі декартових координат. Якщо задано як список, це має бути список списків, кожен з яких містить рядок атома та кортеж координат. | Так      | N/A                                                                      | `[['O', (0, 0, 0)], ['H', (0, 1, 0)], ['H', (0, 0, 1)]]` або `"O 0 0 0; H 0 1 0; H 0 0 1"` |
| backend\_name          | str                                                            | Назва бекенду для виконання запиту.                                                                                                                                                                                                                                                                         | Так      | N/A                                                                      | `ibm_fez`                                                                                 |
| max\_states            | int                                                            | Максимальна розмірність підпростору для діагоналізації. Менша кількість станів використовується, якщо число не є точним квадратом.                                                                                                                                                                          | Так      | N/A                                                                      | `100`                                                                                     |
| max\_expansion\_states | int                                                            | Максимальна кількість класично-генерованих CI-станів, що включаються в кожну ітерацію.                                                                                                                                                                                                                     | Так      | N/A                                                                      | `10`                                                                                      |
| molecule_options                | dict                                                           | Параметри, пов'язані з молекулою, що використовується як вхідні дані для HI-VQE. Докладніші відомості містяться в розділі [Параметри молекули](#molecule-options).                                                                                                                                           | Ні       | Дивись розділ [Параметри молекули](#molecule-options).                   | `{"basis": "sto3g", "unit": "angstrom" }`                               |
| hivqe_options                | dict                                                           | Параметри, що контролюють поведінку алгоритму HI-VQE. Докладніші відомості містяться в розділі [Параметри HI-VQE](#hi-vqe-options).                                                                                                                                                                          | Ні       | Дивись розділ [Параметри HI-VQE](#hi-vqe-options).                       | `{"shots": 10_000, "max_iter": 10 }`                               |
### Параметри молекули
Наступна таблиця містить деталі про всі ключі та значення, які можна задати в словнику `molecule_options`, а також їхні типи даних і значення за замовчуванням. Всі ключі є необов'язковими.

| Ключ                              | Тип значення                        | Значення за замовчуванням        | Допустимий діапазон                                                                                                                                            | Пояснення                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              |
|:----------------------------------|:-----------------------------------:|:--------------------------------:|:---------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `"charge"`                        | `int`                               | `0`                              | Різні                                                                                                                                                          | Ціле число, що вказує повний електричний заряд молекулярної системи. Значення за замовчуванням — 0; однак це може бути будь-яке ціле число.                                                                                                                                                                                                                                                                                                                                                                                             |
| `"basis"`                         | `str`                               | `'sto-3g'`                       | Різні                                                                                                                                                          | Рядок, що задає тип базису; передається до pyscf. (напр.: `"sto-3g"`, `"3-21g"`, `"6-31g"`, `"cc-pvdz"`)                                                                                                                                                                                                                                                                                                                                                                                                                               |
| `"active_orbitals"`               | `List[int]`                         | Кожен індекс орбіталі.           | Індекси просторових орбіталей, допустимі для задачі.                                                                                                           | Список індексів активних орбіталей у діапазоні [0, n), де n — кількість кубітів, що використовуються в задачі. Якщо це задано, аргумент frozen_orbitals також має бути задано.                                                                                                                                                                                                                                                                                                                                                          |
| `"frozen_orbitals"`               | `List[int]`                         | Без індексів.                    | Індекси просторових орбіталей, допустимі для задачі, за винятком активних орбіталей.                                                                           | Список індексів заморожених орбіталей у тому самому діапазоні, що й active_orbitals. Якщо задано, active_orbitals також має бути задано. Зауваж, що заморожувати слід тільки зайняті орбіталі, оскільки кількість активних електронів зменшується на 2 для кожної зайнятої орбіталі, що заморожується.                                                                                                                                                                                                                                   |
| `"orbital_coeffs"`                | `List[List[float]]`                 | Молекулярні орбіталі Гартрі–Фока. | Різні.                                                                                                                                                        | Коефіцієнти для просторових орбіталей, що використовуються при обчисленні інтегралів електронного відштовхування для системи. Деякі допустимі приклади — молекулярні орбіталі Гартрі–Фока, натуральні орбіталі та орбіталі AVAS.                                                                                                                                                                                                                                                                                                      |
| `"symmetry"`                      | `Union[str, bool]`                  | `False`                          | `True` або `False`                                                                                                                                             | Використовується для застосування симетрії точкової групи при початкових молекулярних обчисленнях для побудови адаптованого до симетрії орбітального базису. Ці адаптовані до симетрії орбіталі використовуються як базисні функції для наступних SCF-обчислень. Значення за замовчуванням — False; якщо встановлено True, то симетрія буде застосована і довільні точкові групи будуть автоматично виявлені та використані. Якщо призначена певна симетрія, наприклад symmetry = "Dooh", то буде виникнуто помилку, якщо молекулярна геометрія не має цієї необхідної симетрії. |
| `"symmetry_subgroup"`             | `Optional[str]`                     | `None`                           | Дивись [документацію pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                    | Може використовуватися для генерації підгрупи виявленої симетрії. Не має ефекту при задані симетрії через ключовий аргумент symmetry.                                                                                                                                                                                                                                                                                                                                                                                                   |
| `"unit"`                          | `str`                               | `"angstrom"`                     | Дивись [документацію pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                    | Вказує одиницю вимірювання для атомних координат і відстаней. За замовчуванням використовуються ангстреми.                                                                                                                                                                                                                                                                                                                                                                                                                              |
| `"nucmod"`                        | `Optional[Union[dict, str]]`        | `None`                           | Дивись [документацію pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                    | Вказує ядерну модель, яку слід використовувати. За замовчуванням використовується точкова ядерна модель, а інші значення вмикають гаусівську ядерну модель. Якщо задано функцію, вона буде використовуватися з гаусівською ядерною моделлю для генерації значення розподілу ядерного заряду 'zeta'.                                                                                                                                                                                                                                       |
| `"pseudo"`                        | `Optional[Union[dict, str]]`        | `None`                           | Дивись [документацію pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                    | Вказує псевдопотенціал для атомів у молекулі. За замовчуванням це None, що означає відсутність псевдопотенціалів і явне включення всіх електронів в обчислення.                                                                                                                                                                                                                                                                                                                                                                         |
| `"cart"`                          | `bool`                              | `False`                          | Дивись [документацію pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                    | Вказує, чи використовувати декартові GTO як кутово-імпульсні базисні функції в обчисленні. Значення за замовчуванням False використовуватиме сферичні GTO.                                                                                                                                                                                                                                                                                                                                                                              |
| `"magmom"`                        | `Optional[List[Union[int, float]]]` | `None`                           | Дивись [документацію pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                    | Встановлює коліарний спіновий магнітний момент кожного атома. За замовчуванням це None і кожен атом ініціалізується зі спіном нуль.                                                                                                                                                                                                                                                                                                                                                                                                     |
| `"avas_aolabels"`                 | `Optional[List[str]]`               | `None`                           | напр. ["H 1s", "O 2p"] для H2O                                                                                                                                | Визначає атомні орбіталі, що включаються до схеми AVAS. Дивись [документацію AVAS](https://arxiv.org/abs/1701.07862).                                                                                                                                                                                                                                                                                                                                                                                                                   |
| `"avas_threshold"`                | `float`                             | `0.2`                            | Від 0.0 до 2.0                                                                                                                                                 | Вказує граничне значення, що використовується для визначення, які атомні орбіталі (AO) зберігаються в активному просторі.                                                                                                                                                                                                                                                                                                                                                                                                               |
| `"noons_level"`                   | `Optional[str]`                     | `None`                           | `"mp2"` або `"ccsd"`                                                                                                                                           | Визначає теоретичний підхід для підготовки натуральних орбіталей і вибору активних орбіталей на основі схеми чисел заповнення натуральних орбіталей (NOON). Дивись [документацію NOONs](https://doi.org/10.1063/5.0042147). Обидва індекси активних і заморожених орбіталей мають бути надані для контролю кількості орбіталей (і кількості кубітів).                                                                                                                                                                                      |
### Параметри HI-VQE
Наступна таблиця містить деталі про всі ключі та значення, які можна задати в словнику `hivqe_options`, а також їхні типи даних і значення за замовчуванням. Всі ключі є необов'язковими.

| Ключ                              | Тип значення                        | Значення за замовчуванням        | Допустимий діапазон                                                                                                                                            | Пояснення                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              |
|:----------------------------------|:-----------------------------------:|:--------------------------------:|:---------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `"shots"`                         | `int`                               | `1_000`                          | Від 1 до 10 000.                                                                                                                                               | Кількість знімків для використання на квантовому пристрої за ітерацію.                                                                                                                                                                                                                                                                                                                                                                                                                                                                  |
| `"max_iter"`                      | `int`                               | `25`                             | Від 1 до 50.                                                                                                                                                   | Максимальна кількість ітерацій для оптимізації ансацу. Алгоритм може використовувати менше ітерацій, якщо збіжність досягнута раніше.                                                                                                                                                                                                                                                                                                                                                                                                   |
| `"initial_basis_states"`          | `List[str]`                         | Стан Гартрі–Фока.                | Бінарні рядки з кількістю бітів, що відповідає необхідній кількості кубітів для задачі.                                                                        | Може використовуватися для перезапуску алгоритму з класичними станами з попереднього результату.                                                                                                                                                                                                                                                                                                                                                                                                                                        |
| `"ansatz"`                        | `str`                               | `"epa"`                          | `"epa"`, `"hea"` або `"lucj"`                                                                                                                                  | Вказує квантовий ансац для оптимізації з метою генерації нових станів. `"epa"` вибирає ансац, що зберігає збудження. `"hea"` вибирає апаратно-ефективний ансац. `"lucj"` вибирає локальний унітарний кластерний ансац Жастрова.                                                                                                                                                                                                                                                                                                         |
| `"convergence_count"`             | `int`                               | `3`                              | Щонайменше 2.                                                                                                                                                  | Кількість ітерацій без суттєвої зміни обчисленої енергії, що мають пройти до того, як алгоритм вважатиметься збіжним.                                                                                                                                                                                                                                                                                                                                                                                                                   |
| `"convergence_abstol"`            | `float`                             | `1e-4`                           | Більше 0 і щонайбільше 1.                                                                                                                                      | Величина зміни обчисленої енергії, що вважається суттєвою для перевірки збіжності.                                                                                                                                                                                                                                                                                                                                                                                                                                                     |
| `"reset_convergence_count"`       | `bool`                              | `True`                           | `True` або `False`                                                                                                                                             | Якщо `True`, `convergence_count` ітерацій мають відбутися без переривання суттєвою зміною, щоб кваліфікуватися як збіжність. Якщо `False`, алгоритм зупиниться після `convergence_count`, якщо несуттєві зміни відбувались в будь-яких ітераціях під час процесу оптимізації.                                                                                                                                                                                                                                                          |
| `"configuration_recovery"`        | `bool`                              | `True`                           | `True` або `False`.                                                                                                                                            | Чи використовувати відновлення конфігурацій з пакету `qiskit-addon-sqd`. Якщо True, недійсні стани, відібрані з квантового пристрою, коригуються класично. Якщо False, вони відкидаються.                                                                                                                                                                                                                                                                                                                                               |
| `"ansatz_entanglement"`           | `str`                               | `"circular"`                     | Будь-яке з `"linear"`, `"reverse_linear"`, `"pairwise"`, `"circular"`, `"full"` або `"sca"`. При використанні ансацу `"lucj"` також доступний `"lucj_default"`. | Вказує схему заплутування, яку слід використовувати в квантовій схемі, відповідно до загальних конвенцій Qiskit та [конвенцій ffsim для ансацу LUCJ](https://qiskit-community.github.io/ffsim/how-to-guides/simulate-lucj.html).                                                                                                                                                                                                                                                                                                        |
| `"ansatz_reps"`                   | `int`                               | `2`                              | Більше 0.                                                                                                                                                      | Кількість повторень кожного шару в квантовій схемі.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     |
| `"amplitude_screening_tolerance"` | `Union[float,int]`                  | `0`                              | Щонайменше 0 і менше 1.                                                                                                                                        | Допуск для вирішення, які стани слід відсіяти з підпростору після діагоналізації. Визначає поріг включення для станів підпростору на основі їхніх обчислених амплітуд.                                                                                                                                                                                                                                                                                                                                                                  |
| `"overlap_screening_tolerance"`   | `float`                             | `1e-2`                           | Від `1e-4` до `1e-1` включно.                                                                                                                                  | Допуск для прогнозування, які стани слід відсіяти з підпростору перед діагоналізацією. Контролює точність прогнозованих амплітуд для кожного стану: менше значення дає точніші прогнози.                                                                                                                                                                                                                                                                                                                                                |
## Вихідні дані
Функція повертає словник з чотирма ключами та значеннями. Ключі та значення задокументовані в наступній таблиці:

| Ключ            | Тип значення  | Пояснення                                                                                                               |
|:----------------|---------------|---------------------------------------------------------------------------------------------------------------------------|
| `"energy"`      | `float`       | Апроксимована енергія основного стану молекули.                                                                          |
| `"states"`      | `List[str]`   | Вибрані детермінанти, що утворюють підпростір для знаходження енергії. Вони представлені у форматі альфа-бета, що чергується. |
| `"eigenvector"` | `List[float]` | Власний вектор, що відповідає основному стану підпростору, складеного з `"states"`.                                      |
| `"energy_variance"` | `float` | Дисперсія енергії основного стану підпростору, складеного з `"states"`, що дає уявлення про якість розв'язку. Це значення невід'ємне, і менше значення означає, що основний стан підпростору точніше апроксимує власний стан гамільтоніана системи. |
| `"energy_history"` | `List[float]` | Енергії, обчислені на кожній ітерації під час гібридного процесу оптимізації, у порядку їх обчислення. За ітерацію обчислюються дві енергії як частина процесу оптимізації SPSA. |
## Приклад
Перший приклад показує, як обчислити енергію основного стану молекули NH3 за допомогою алгоритму HI-VQE.
#### Визначення молекулярної геометрії та параметрів
Молекулярна геометрія NH3 задається декартовими координатами, розділеними символом ";" для кожного атома.

In [3]:
# Define the molecule geometry
geometry = """
N         -0.85188       -0.02741        0.03141;
H          0.16545        0.00593       -0.01648;
H         -1.16348       -0.39357       -0.86702;
H         -1.16348        0.94228        0.06281;
"""

Додаткові параметри можна визначити та надати для молекулярної системи у наступному форматі словника.

In [4]:
# Configure some options for the job.
molecule_options = {"basis": "sto3g"}
hivqe_options = {"shots": 100, "max_iter": 20}

Виконай функцію з вхідними даними геометрії та параметрів.

In [5]:
# Run HI-VQE
job = function.run(
    geometry=geometry,
    # `backend_name` is the name of a backend with at least 16 qubits, for example, "ibm_marrakesh".
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=10,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

Варто роздрукувати ідентифікатор завдання функції, щоб його можна було надати у запитах на підтримку у разі виникнення проблем.

In [6]:
print("Job ID:", job.job_id)

Job ID: 128ee399-7cfc-4be2-91e9-c4abd22b97c7


This example then utilizes 16 qubits with 8 orbitals of sto3g basis for an NH3 molecule.

Check your Qiskit Function workload's [status](/docs/guides/functions#check-job-status) or return [results](/docs/guides/functions#retrieve-results) as follows:

In [7]:
print(job.status())

QUEUED


Цей приклад використовує 16 кубітів з 8 орбіталями базису sto3g для молекули NH3.
Перевіряй [статус](/guides/functions#check-job-status) свого навантаження функції Qiskit або отримуй [результати](/guides/functions#retrieve-results) таким чином:

In [8]:
result = job.result()

# Output can be long, so we display a shortened representation
shortened_result = reprlib.repr(result)
print(shortened_result)

{'eigenvector': [0.9824200343205695, 0.009520846167419264, 6.365368845740147e-08, 3.6832123006425615e-07, 0.0012869941182066654, 2.3403891050875465e-05, ...], 'energy': -55.521146537970466, 'energy_history': [-55.52091922342852, -55.52113695367561, -55.521146537970466, -55.52114653864798, -55.521146537970466, -55.521146537970466, ...], 'energy_variance': 9.788555455342562e-12, ...}


To access the ground state energy, use the "energy" key. The "eigenvector" key provides the CI coefficients with corresponding bitstring notation of electron configuration stored with "states" of the results.

In [10]:
fci_energy = -55.521148034704126  # the exact energy using FCI method
hivqe_energy = result["energy"]
print(
    f"|Exact Energy - HI-VQE Energy|: {abs(fci_energy - hivqe_energy) * 1000} mHa"
)
print(f"Sampled Number of States: {len(result['states'])}")

|Exact Energy - HI-VQE Energy|: 0.0014967336596782843 mHa
Sampled Number of States: 1936


Після завершення завдання результати можна отримати за допомогою екземпляра `result()`.

In [11]:
# This cell is hidden from users
backend_name = service.least_busy(operational=True, min_num_qubits=38).name

In [12]:
# Define Li2S geometries
Li2S_geoms = {
    "Li2S_1.51": "S        -1.239044    0.671232   -0.030374;Li       -1.506327    0.432403   -1.498949;Li       -0.899996    0.973348    1.826768;",
    "Li2S_2.40": "S        -1.741432    0.680397    0.346702;Li       -0.529307    0.488006   -1.729343;Li       -1.284307    0.989409    2.177209;",
    "Li2S_3.80": "S        -2.707255    0.674298    0.909161;Li        0.079218    0.552012   -1.671656;Li       -0.927010    0.931502    1.557063;",
}

# Configure some options for the job.
molecule_options = {
    "basis": "sto3g",
}
hivqe_options = {
    "shots": 100,
    "max_iter": 20,
}

results = []
for geom in ["Li2S_1.51", "Li2S_2.40", "Li2S_3.80"]:
    # Run HI-VQE
    job = function.run(
        geometry=Li2S_geoms[geom],
        backend_name=backend_name,  # can use any device with at least 38 qubits
        max_states=2000,
        max_expansion_states=10,
        molecule_options=molecule_options,
        hivqe_options=hivqe_options,
    )
    results.append(job.result())

Для доступу до енергії основного стану використовуй ключ "energy". Ключ "eigenvector" надає CI-коефіцієнти з відповідними позначеннями конфігурацій електронів у вигляді бітових рядків, збережених у "states" результатів.

In [13]:
job = function.run(
    geometry="invalid-geometry",  # This will cause an error
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=15,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

job.result()

QiskitServerlessException: {"message": "An unexpected error occurred during job execution. Please make sure that your inputs are valid. If you are still experiencing problems, you can contact the Qunova Computing support service at qiskit.support@qunovacomputing.com and provide the Function job ID of this job for more assistance.", "status": "failure"}

In [14]:
job.status()

'ERROR'

Вихідні дані:

|Exact Energy - HI-VQE Energy|: 0.077237504 mHa
Sampled Number of States: 1936
## Продуктивність
У цьому розділі наведено продемонстровані еталонні обчислення HI-VQE для випадку з 24 кубітами для Li2S, випадку з 40 кубітами для молекули N2 та випадку з 44 кубітами для системи FeP-NO.
#### Крива поверхні дисоціативної потенційної енергії для молекули Li2S з 24 кубітами
Крива PES показана з референсом FCI та початковим наближенням RHF, а також похибкою енергії від референсу FCI.

![Зображення, що показує, що HI-VQE дає розв'язки в межах хімічної точності класичної референсної кривої PES для системи Li2S.](../docs/images/guides/qunova-chemistry/Li2S_PES.avif)

Обчислення проводилися з такими геометріями та параметрами.